# NB13 — Calibration (ECE / Brier / Temperature scaling)

Computes ECE (15-bin) and Brier score for the proposed model and the vanilla baseline on **Ben-Sarc
binary**, then fits a single temperature on the **validation** logits and re-measures (macro-F1 is
unchanged by temperature scaling). Saves a reliability diagram. CPU/Mac.


In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
def find_repo_root():
    for c in [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents):
        if (c/"01_data").exists(): return c.resolve()
    raise RuntimeError("repo root not found.")
ROOT=find_repo_root(); TABLES=ROOT/"04_outputs"/"tables"; PRED=ROOT/"04_outputs"/"predictions"; FIGS=ROOT/"04_outputs"/"figures"
for p in [TABLES,FIGS]: p.mkdir(parents=True,exist_ok=True)
TASK="ben_sarc_binary"
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
def collect(task, split):
    out={}
    for f in sorted(PRED.glob(f"*_{task}_{split}_predictions.csv")):
        name=f.name.replace(f"_{task}_{split}_predictions.csv","")
        try: out[name]=pd.read_csv(f).dropna(subset=["text"]).drop_duplicates("text")
        except Exception as e: print("skip",f.name,e)
    return out
def macro(df): return f1_score(df["gold_label"],df["pred_label"],average="macro",zero_division=0)
print("ROOT:",ROOT)

ROOT: /Users/sefayet/Desktop/Github/Sarcasm_detection


In [2]:
import torch, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
test=collect(TASK,"test"); val=collect(TASK,"val")
def logit_cols(df): return sorted([c for c in df.columns if c.startswith("logit_")])
def probs_from(df):
    L=df[logit_cols(df)].values; e=np.exp(L-L.max(1,keepdims=True)); return e/e.sum(1,keepdims=True)
def ece(p,y,nb=15):
    conf=p.max(1); pred=p.argmax(1); acc=(pred==y).astype(float)
    bins=np.linspace(0,1,nb+1); e=0.0
    for i in range(nb):
        m=(conf>bins[i])&(conf<=bins[i+1])
        if m.sum()>0: e+=abs(acc[m].mean()-conf[m].mean())*m.mean()
    return float(e)
def brier(p,y):
    oh=np.eye(p.shape[1])[y]; return float(((p-oh)**2).sum(1).mean())
def fit_T(logits,y):
    t=torch.nn.Parameter(torch.ones(1)); lg=torch.tensor(logits,dtype=torch.float); ly=torch.tensor(y)
    opt=torch.optim.LBFGS([t],lr=0.05,max_iter=200)
    def cl(): opt.zero_grad(); l=F.cross_entropy(lg/t.clamp(min=1e-3),ly); l.backward(); return l
    opt.step(cl); return float(t.item())

In [3]:
targets=[m for m in ["09b_fgm_awp","05_baseline_banglabert"] if m in test]
rows=[]; rel={}
for m in targets:
    dt=test[m]; y=dt["gold_label"].values; Lt=dt[logit_cols(dt)].values; pt=probs_from(dt)
    pre={"model":m,"ece":ece(pt,y),"brier":brier(pt,y),"macro_f1":macro(dt)}
    T=1.0
    if m in val:
        dv=val[m]; T=fit_T(dv[logit_cols(dv)].values, dv["gold_label"].values)
        Lt_s=Lt/T; e=np.exp(Lt_s-Lt_s.max(1,keepdims=True)); pts=e/e.sum(1,keepdims=True)
        pre.update({"temperature":T,"ece_after":ece(pts,y),"brier_after":brier(pts,y)})
        rel[m]=(pt,pts,y)
    else:
        pre.update({"temperature":None,"ece_after":None,"brier_after":None}); rel[m]=(pt,None,y)
    rows.append(pre)
res=pd.DataFrame(rows); res.to_csv(TABLES/"13_calibration.csv",index=False)
print("="*80); print("  CALIBRATION — Ben-Sarc binary"); print("="*80)
print(res.to_string(index=False,float_format="%.4f"))

# reliability diagram for the proposed model
m=targets[0]; pt,pts,y=rel[m]
plt.figure(figsize=(5,5))
for p,lab in [(pt,"before"),(pts,"after T")]:
    if p is None: continue
    conf=p.max(1); pred=p.argmax(1); acc=(pred==y).astype(float)
    bins=np.linspace(0,1,16); xs=[]; ys=[]
    for i in range(15):
        mk=(conf>bins[i])&(conf<=bins[i+1])
        if mk.sum()>0: xs.append(conf[mk].mean()); ys.append(acc[mk].mean())
    plt.plot(xs,ys,"o-",label=lab)
plt.plot([0,1],[0,1],"k--",alpha=0.5,label="perfect")
plt.xlabel("confidence"); plt.ylabel("accuracy"); plt.title(f"Reliability — {m}"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIGS/"13_reliability.png",dpi=160); plt.close()
print("saved -> 04_outputs/figures/13_reliability.png")

  CALIBRATION — Ben-Sarc binary
                 model    ece  brier  macro_f1  temperature  ece_after  brier_after
           09b_fgm_awp 0.0219 0.2791    0.8038       0.9595     0.0210       0.2787
05_baseline_banglabert 0.0550 0.2884    0.8025       1.3293     0.0192       0.2824
saved -> 04_outputs/figures/13_reliability.png
